# 04 — Difference-in-Differences

Goal: Estimate causal effect of the design-change rollout using DiD.

Framing: Variant B was rolled out to the Southeast region first (Jan 2023).
Other regions serve as control.

Key steps:
1. Check parallel trends assumption (REQUIRED — not assumed)
2. OLS with unit + time fixed effects
3. Placebo test: pretend treatment happened earlier; effect should be zero
4. Interpret coefficient as ATT for the treated region

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
import sys; sys.path.insert(0, '..')
from src.diagnostics import plot_parallel_trends

df = pd.read_csv('../data/field_panel.csv', parse_dates=['install_date'])
df['install_month_num'] = df.install_date.dt.month + (df.install_date.dt.year - 2022) * 12

# For DiD: frame as 'Southeast got the redesign in Jan 2023'
df['treated_group'] = (df.region == 'Southeast').astype(int)
df['post_period']   = (df.install_month_num >= 13).astype(int)  # month 13 = Jan 2023

%matplotlib inline

In [ ]:
# Parallel trends check
trend = df.groupby(['install_month_num', 'treated_group']).failure_event.mean().reset_index()
fig, ax = plt.subplots(figsize=(9,4))
for grp, label, color in [(1,'Southeast (treated)','steelblue'), (0,'Other regions (control)','tomato')]:
    sub = trend[trend.treated_group == grp]
    ax.plot(sub.install_month_num, sub.failure_event * 100, label=label, color=color, linewidth=2, marker='o')
ax.axvline(13, color='gold', linestyle='--', label='Treatment start (Jan 2023)')
ax.set_title('Parallel Trends Check — Pre-treatment trends should be parallel')
ax.set_xlabel('Install month'); ax.set_ylabel('Monthly failure rate (%)')
ax.legend()
plt.savefig('../figures/parallel_trends.png', dpi=150, bbox_inches='tight')
plt.show()
print('Pre-treatment (months 1-12): are the two lines parallel? Check visually AND statistically.')

In [ ]:
# DiD OLS with interaction term
model = smf.ols(
    'failure_event ~ treated_group * post_period + C(region) + C(install_month_num)',
    data=df
).fit(cov_type='HC1')

coef   = model.params.get('treated_group:post_period', np.nan)
se     = model.bse.get('treated_group:post_period', np.nan)
pval   = model.pvalues.get('treated_group:post_period', np.nan)
ci     = model.conf_int().loc['treated_group:post_period'].tolist() if 'treated_group:post_period' in model.params else [None, None]

print(f'DiD ATT: {coef*100:+.2f} percentage points (SE: {se*100:.2f})')
print(f'95% CI: [{ci[0]*100:.2f}, {ci[1]*100:.2f}]')
print(f'p-value: {pval:.4f}')

In [ ]:
# Placebo test: pretend treatment started at month 7 (pre-treatment)
df['post_placebo'] = (df.install_month_num >= 7).astype(int)
placebo_model = smf.ols(
    'failure_event ~ treated_group * post_placebo + C(region) + C(install_month_num)',
    data=df[df.install_month_num < 13]  # pre-treatment window only
).fit(cov_type='HC1')

placebo_coef = placebo_model.params.get('treated_group:post_placebo', np.nan)
print(f'\nPlacebo test coefficient: {placebo_coef*100:+.2f}%')
print(f'Expected: near zero. If significant: parallel-trends assumption is violated.')